In [1]:
import os
import rasterio
from importlib_resources import files
from pathlib import Path
from tqdm import tqdm

from beak.experimental.raster_processing import unify_raster_grids
from beak.experimental.io import save_raster, create_file_list, check_path


**User inputs**
1. Select the root folder where the rasters are stored.
2. Then, go down to select the subfolders that contain the rasters to be unified.<p>

The script will search for all rasters and store them in relative paths.

Original **YTU** were just copied from the RAW folder to the PROCESSED folder, since they are the blueprint for the base raster and thus do not need to be unified.

In [2]:
BASE_PATH = files("beak.data")

EPSG_CODE = 102008
RESOLUTION = 500
BASE_SPATIAL = str(EPSG_CODE) + "_" + str(RESOLUTION)
BASE_EXTENT = "mama_nico_upmidwest"
BASE_RASTER = BASE_PATH / "BASE_RASTERS" / str("EPSG_" + str(EPSG_CODE) + "_RES_" + str(RESOLUTION) + "_" + BASE_EXTENT + ".tif")
BASE_EVIDENCE = "geophysics"

# Export
# Some data sets are already **LOG** scaled and need special actions. They will be unified and stored in a separate folder.
PATH_INPUT = BASE_PATH / "RAW" / BASE_EVIDENCE / "national_scale"

PATH_ROOT = BASE_PATH / "PROCESSED" / str("regional" + "_" + BASE_EXTENT + "_" + BASE_SPATIAL)
PATH_EXPORT = PATH_ROOT / "unified" / BASE_EVIDENCE
PATH_EXPORT_LOG = PATH_ROOT / "unified_scaled_log" / BASE_EVIDENCE

CURRENT_DIR = Path(os.getcwd()) / PATH_EXPORT.name
OUT_FOLDER = PATH_EXPORT
OUT_FOLDER_LOG = PATH_EXPORT_LOG

print(f"Input folder: {PATH_INPUT}")
print(f"Export folder: {OUT_FOLDER}")

Input folder: S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\RAW\geophysics\national_scale
Export folder: S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\PROCESSED\regional_mama_nico_upmidwest_102008_500\unified\geophysics


Select subfolders to be scaled.

In [3]:
for folder in os.listdir(PATH_INPUT):
  if os.path.isdir(os.path.join(PATH_INPUT, folder)):
    print(folder)


gravity
magnetic
magnetotelluric
radiometric
seismic
unsorted


In [4]:
SELECTION = ["gravity", 
             "magnetic",
             "magnetotelluric",
             "radiometric",
             "seismic"]

input_folders = [PATH_INPUT / folder for folder in SELECTION]

print("Selected folders:")
for folder in input_folders:
  print(folder)

Selected folders:
S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\RAW\geophysics\national_scale\gravity
S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\RAW\geophysics\national_scale\magnetic
S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\RAW\geophysics\national_scale\magnetotelluric
S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\RAW\geophysics\national_scale\radiometric
S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\RAW\geophysics\national_scale\seismic


**Select files**

In [5]:
# Create file list
file_list = []
for folder in input_folders:
  files = create_file_list(folder, recursive=True)
  file_list.extend(files)

# Remove CONUS 2021 data
file_list = [file for file in file_list if "CONUS_2021" not in str(file)]

# Create file list for log scaled data
log_files = [file.stem for file in file_list if "Log" in file.stem]

# Show results
print(f"Found {len(file_list)} files including {len(log_files)} log scaled files.")


Found 25 files including 1 log scaled files.


**Run unification**

In [6]:
# Select to write output file
dry_run = False

if dry_run:
  print("Dry run, no files will be written.\n")

# Unify
for file in tqdm(file_list, total=len(file_list)):
    folder_relative = os.path.relpath(file.parent, PATH_INPUT)

    raster = rasterio.open(file)
    unified_raster, unified_meta = unify_raster_grids(BASE_RASTER, [file], resampling_mode="auto", same_extent=True, same_shape=True)[0]

    if not dry_run:
      if file.stem in log_files:
        log_folder = OUT_FOLDER_LOG / str(folder_relative)
        log_out_path = log_folder / file.name
        check_path(Path(os.path.dirname(log_out_path)))
        save_raster(log_out_path, array=unified_raster, dtype="float32", metadata=unified_meta)
      else:
        output_folder = OUT_FOLDER / str(folder_relative)
        out_path = output_folder / file.name
        check_path(Path(os.path.dirname(out_path)))
        save_raster(out_path, array=unified_raster, dtype="float32", metadata=unified_meta)


  4%|▍         | 1/25 [00:00<00:10,  2.36it/s]

File already exists: Gravity.tif.


  8%|▊         | 2/25 [00:00<00:08,  2.58it/s]

File already exists: Gravity_HGM.tif.


 12%|█▏        | 3/25 [00:01<00:08,  2.61it/s]

...

File already exists: LAB.tif.


 96%|█████████▌| 24/25 [00:12<00:00,  3.03it/s]

File already exists: LAB_HGM.tif.


100%|██████████| 25/25 [00:12<00:00,  2.01it/s]

File already exists: Moho.tif.
